In [1]:
%pip install "psycopg2-binary"

Note: you may need to restart the kernel to use updated packages.


CONECTÁNDOSE A NEON

In [13]:
import os
import psycopg2
from dotenv import load_dotenv

# Cargar las variables del archivo .env
load_dotenv()

# Obtener la cadena de conexión
neon_url = os.getenv("NEON_DATABASE_URL")

if not neon_url:
    raise ValueError("Falta configurar NEON_DATABASE_URL en el archivo .env")

# Conectar a Neon
conn = psycopg2.connect(neon_url)

try:
    with conn.cursor() as cur:
        # Verificar la conexión
        cur.execute("SELECT current_database(), current_schema();")
        print("Base de datos y esquema:", cur.fetchone())

        # Consultar las tablas y columnas del proyecto
        cur.execute("""
            SELECT table_name, column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name IN (
                  'incidentes',
                  'hipotesis',
                  'soluciones',
                  'adjuntos_incidente',
                  'validaciones'
              )
            ORDER BY table_name, ordinal_position;
        """)

        columnas = cur.fetchall()

        print("\nTablas y columnas encontradas:")
        for tabla, columna, tipo in columnas:
            print(f"{tabla}.{columna} ({tipo})")

finally:
    conn.close()


Base de datos y esquema: ('neondb', 'public')

Tablas y columnas encontradas:
adjuntos_incidente.id (uuid)
adjuntos_incidente.id_incidente (uuid)
adjuntos_incidente.nombre_archivo (character varying)
adjuntos_incidente.ubicacion_archivo (text)
adjuntos_incidente.texto_extraido (text)
adjuntos_incidente.cargado_por_id (character varying)
adjuntos_incidente.fecha_creacion (timestamp with time zone)
hipotesis.id (uuid)
hipotesis.id_incidente (uuid)
hipotesis.descripcion (text)
hipotesis.evidencia (text)
hipotesis.estado (character varying)
hipotesis.propuesto_por_id (character varying)
hipotesis.evaluado_por_id (character varying)
hipotesis.fecha_creacion (timestamp with time zone)
incidentes.id (uuid)
incidentes.codigo (character varying)
incidentes.titulo (character varying)
incidentes.descripcion (text)
incidentes.componente (character varying)
incidentes.mensaje_error (text)
incidentes.estado (character varying)
incidentes.reportado_por_id (character varying)
incidentes.fecha_creacion

Verifica qué tablas y columnas tienes en Neon

In [14]:

import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2 import sql

load_dotenv()

conn = psycopg2.connect(os.getenv("NEON_DATABASE_URL"))

tablas_esperadas = [
    "incidentes",
    "hipotesis",
    "soluciones",
    "adjuntos_incidente"
]

with conn.cursor() as cur:
    cur.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_type = 'BASE TABLE'
        ORDER BY table_name;
    """)
    tablas_existentes = [fila[0] for fila in cur.fetchall()]

print("Tablas encontradas:", tablas_existentes)

for tabla in tablas_esperadas:
    if tabla in tablas_existentes:
        consulta = sql.SQL("SELECT * FROM {}.{} LIMIT 5").format(
            sql.Identifier("public"),
            sql.Identifier(tabla)
        )
        df = pd.read_sql_query(consulta.as_string(conn), conn)
        print(f"\nTabla: {tabla}")
        display(df)



Tablas encontradas: ['adjuntos_incidente', 'hipotesis', 'history', 'incidentes', 'soluciones']

Tabla: incidentes


C:\Users\PEDRO VARGAS\AppData\Local\Temp\ipykernel_26564\463782908.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(consulta.as_string(conn), conn)


,id,codigo,titulo,descripcion,componente,mensaje_error,estado,reportado_por_id,fecha_creacion
0,10000000-0000-4000-8000-000000000001,INC-0001,Timeout en API de pagos,Las solicitudes de pago exceden el tiempo máxi...,API de pagos,Read timed out,resuelto,A00001,2026-09-04 19:35:55.163786+00:00
1,10000000-0000-4000-8000-000000000002,INC-0002,Error de autenticación OAuth,Un consumidor de la API no puede autenticarse ...,Autenticación,401 Unauthorized,resuelto,A00002,2026-07-24 12:16:28.400328+00:00
2,10000000-0000-4000-8000-000000000003,INC-0003,Certificado TLS vencido,Una comunicación entre servicios falla por un ...,Conectividad segura,Certificate expired,resuelto,A00003,2026-08-14 20:01:58.062031+00:00
3,10000000-0000-4000-8000-000000000004,INC-0004,Pool de conexiones agotado,La aplicación no consigue conexiones disponibl...,Base de datos,Connection pool exhausted,investigando,A00001,2026-07-26 06:32:55.641741+00:00
4,10000000-0000-4000-8000-000000000005,INC-0005,Acumulación de mensajes en cola,La cola de mensajería acumula mensajes y aumen...,Mensajería,Consumer lag above threshold,investigando,A00002,2026-07-22 04:19:55.230139+00:00


C:\Users\PEDRO VARGAS\AppData\Local\Temp\ipykernel_26564\463782908.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(consulta.as_string(conn), conn)



Tabla: hipotesis


...


Tabla: soluciones


,id,id_incidente,id_hipotesis,descripcion,estado,resultado,propuesto_por_id,aplicado_por_id,validado_por_id,fecha_creacion
0,30000000-0000-4000-8000-000000000001,10000000-0000-4000-8000-000000000001,20000000-0000-4000-8000-000000000001,Ajustar el tiempo de espera del cliente dentro...,aplicada,efectiva,A00001,A00002,A00003,2026-09-08 00:17:58.867238+00:00
1,30000000-0000-4000-8000-000000000002,10000000-0000-4000-8000-000000000001,20000000-0000-4000-8000-000000000002,"Revisar la capacidad del servicio, las métrica...",propuesta,pendiente,A00002,None,None,2026-09-09 01:57:02.695598+00:00
2,30000000-0000-4000-8000-000000000003,10000000-0000-4000-8000-000000000002,20000000-0000-4000-8000-000000000003,Renovar el token mediante el flujo de autentic...,aplicada,efectiva,A00003,A00001,A00002,2026-07-29 07:01:46.950845+00:00
3,30000000-0000-4000-8000-000000000004,10000000-0000-4000-8000-000000000002,20000000-0000-4000-8000-000000000004,Modificar la audiencia del token sin evidencia...,retirada,no_efectiva,A00001,A00002,A00003,2026-07-28 15:18:32.980147+00:00
4,30000000-0000-4000-8000-000000000005,10000000-0000-4000-8000-000000000003,20000000-0000-4000-8000-000000000005,Renovar el certificado TLS mediante el procedi...,aplicada,efectiva,A00002,A00003,A00001,2026-08-20 13:43:05.946583+00:00


C:\Users\PEDRO VARGAS\AppData\Local\Temp\ipykernel_26564\463782908.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(consulta.as_string(conn), conn)



Tabla: adjuntos_incidente


,id,id_incidente,nombre_archivo,ubicacion_archivo,texto_extraido,cargado_por_id,fecha_creacion
0,40000000-0000-4000-8000-000000000001,10000000-0000-4000-8000-000000000001,logs_timeout_api.txt,test-data/incidentes/INC-0001/logs_timeout_api...,Ejemplo de log: solicitud a API de pagos super...,A00001,2026-09-05 08:38:10.390635+00:00
1,40000000-0000-4000-8000-000000000002,10000000-0000-4000-8000-000000000002,error_oauth.json,test-data/incidentes/INC-0002/error_oauth.json,Ejemplo de respuesta: 401 Unauthorized. Revisa...,A00002,2026-07-26 00:42:35.309538+00:00
2,40000000-0000-4000-8000-000000000003,10000000-0000-4000-8000-000000000003,detalle_certificado.txt,test-data/incidentes/INC-0003/detalle_certific...,Ejemplo de diagnóstico: certificado TLS expirado.,A00003,2026-08-17 17:31:41.213057+00:00
3,40000000-0000-4000-8000-000000000004,10000000-0000-4000-8000-000000000004,metricas_pool.csv,test-data/incidentes/INC-0004/metricas_pool.csv,Ejemplo de métricas: conexiones activas elevad...,A00001,2026-07-27 10:52:06.989665+00:00
4,40000000-0000-4000-8000-000000000005,10000000-0000-4000-8000-000000000005,metricas_cola.txt,test-data/incidentes/INC-0005/metricas_cola.txt,Ejemplo de diagnóstico: retraso de consumidore...,A00002,2026-07-23 17:01:03.907509+00:00


Leer y unir las cuatro tablas desde Python

In [8]:
import pandas as pd

consulta = """
SELECT
    i.id::text AS incidente_id,
    i.codigo,
    i.titulo,
    i.descripcion,
    i.componente,
    i.mensaje_error,
    i.estado AS estado_incidente,
    i.fecha_creacion,

    COALESCE(h.hipotesis_texto, '') AS hipotesis,
    COALESCE(s.soluciones_texto, '') AS soluciones,
    COALESCE(a.adjuntos_texto, '') AS adjuntos

FROM incidentes i

LEFT JOIN LATERAL (
    SELECT STRING_AGG(
        CONCAT(
            'Hipótesis: ', h.descripcion,
            E'\\nEstado: ', h.estado,
            E'\\nEvidencia: ', COALESCE(h.evidencia, 'No registrada')
        ),
        E'\\n\\n'
    ) AS hipotesis_texto
    FROM hipotesis h
    WHERE h.id_incidente = i.id
) h ON TRUE

LEFT JOIN LATERAL (
    SELECT STRING_AGG(
        CONCAT(
            'Solución: ', s.descripcion,
            E'\\nEstado: ', s.estado,
            E'\\nResultado: ', s.resultado,
            E'\\nHipótesis relacionada: ',
            COALESCE(h2.descripcion, 'No especificada')
        ),
        E'\\n\\n'
    ) AS soluciones_texto
    FROM soluciones s
    LEFT JOIN hipotesis h2
        ON h2.id = s.id_hipotesis
    WHERE s.id_incidente = i.id
) s ON TRUE

LEFT JOIN LATERAL (
    SELECT STRING_AGG(
        CONCAT(
            'Archivo: ', a.nombre_archivo,
            E'\\nTexto extraído: ',
            COALESCE(a.texto_extraido, 'Sin texto extraído')
        ),
        E'\\n\\n'
    ) AS adjuntos_texto
    FROM adjuntos_incidente a
    WHERE a.id_incidente = i.id
) a ON TRUE

ORDER BY i.fecha_creacion DESC;
"""

df_incidentes = pd.read_sql_query(consulta, conn)

print("Total de incidentes:", len(df_incidentes))
print("Columnas:", df_incidentes.columns.tolist())

display(df_incidentes.head(3))


C:\Users\PEDRO VARGAS\AppData\Local\Temp\ipykernel_26564\2180498313.py:66: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_incidentes = pd.read_sql_query(consulta, conn)


Total de incidentes: 15
Columnas: ['incidente_id', 'codigo', 'titulo', 'descripcion', 'componente', 'mensaje_error', 'estado_incidente', 'fecha_creacion', 'hipotesis', 'soluciones', 'adjuntos']


,incidente_id,codigo,titulo,descripcion,componente,mensaje_error,estado_incidente,fecha_creacion,hipotesis,soluciones,adjuntos
0,10000000-0000-4000-8000-000000000012,INC-0012,Fallo de resolución DNS,Una aplicación no puede resolver el nombre de ...,Red,UnknownHostException,resuelto,2026-10-07 04:39:58.745468+00:00,Hipótesis: El registro DNS del servicio podría...,Solución: Verificar la resolución DNS desde el...,
1,10000000-0000-4000-8000-000000000013,INC-0013,Error 403 al acceder a un recurso,Una cuenta de servicio recibe una respuesta 40...,Autorización,403 Forbidden,investigando,2026-10-05 11:34:05.370556+00:00,Hipótesis: La identidad de servicio podría car...,Solución: Revisar los permisos efectivos de la...,
2,10000000-0000-4000-8000-000000000015,INC-0015,Latencia elevada entre microservicios,Las llamadas encadenadas entre servicios prese...,Microservicios,Upstream request timeout,investigando,2026-09-22 23:05:24.728652+00:00,Hipótesis: La latencia podría acumularse debid...,Solución: Analizar las trazas distribuidas y l...,


Construir los documentos de conocimiento

In [9]:

def construir_documento(fila):
    partes = [
        f"Código: {fila['codigo']}",
        f"Título: {fila['titulo']}",
        f"Componente: {fila['componente'] or 'No especificado'}",
        f"Descripción: {fila['descripcion']}",
        f"Mensaje de error: {fila['mensaje_error'] or 'No registrado'}",
        f"Estado del incidente: {fila['estado_incidente']}",
        f"Hipótesis:\n{fila['hipotesis'] or 'Sin hipótesis registrada'}",
        f"Soluciones:\n{fila['soluciones'] or 'Sin soluciones registradas'}",
        f"Adjuntos:\n{fila['adjuntos'] or 'Sin adjuntos registrados'}"
    ]

    return "\n\n".join(partes)


df_incidentes["documento"] = df_incidentes.apply(
    construir_documento,
    axis=1
)

print("Ejemplo de documento que se preparará para Qdrant:\n")
print(df_incidentes.iloc[0]["documento"]
      if not df_incidentes.empty
      else "No hay incidentes registrados.")


Ejemplo de documento que se preparará para Qdrant:

Código: INC-0012

Título: Fallo de resolución DNS

Componente: Red

Descripción: Una aplicación no puede resolver el nombre de un servicio dependiente.

Mensaje de error: UnknownHostException

Estado del incidente: resuelto

Hipótesis:
Hipótesis: El registro DNS del servicio podría ser incorrecto o no estar disponible.
Estado: confirmada
Evidencia: La resolución del nombre falla desde el ambiente afectado.

Soluciones:
Solución: Verificar la resolución DNS desde el ambiente afectado y corregir el registro o la configuración pertinente.
Estado: aplicada
Resultado: efectiva
Hipótesis relacionada: El registro DNS del servicio podría ser incorrecto o no estar disponible.

Adjuntos:
Sin adjuntos registrados


Verificar la conexión y la colección de Qdrant

In [16]:

import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "tarea_final_ia" # os.getenv("QDRANT_COLLECTION_NAME")

if not all([QDRANT_URL, QDRANT_API_KEY, COLLECTION_NAME]):
    raise ValueError(
        "Verifica QDRANT_URL, QDRANT_API_KEY y "
        "QDRANT_COLLECTION_NAME en tu archivo .env"
    )

qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

print("Conexión con Qdrant establecida.")

colecciones = qdrant_client.get_collections()

print("\nColecciones existentes:")
for coleccion in colecciones.collections:
    print("-", coleccion.name)

print("\nColección configurada:", COLLECTION_NAME)

if qdrant_client.collection_exists(COLLECTION_NAME):
    info = qdrant_client.get_collection(COLLECTION_NAME)
    print("La colección ya existe.")
    print("Puntos almacenados:", info.points_count)
    print("Dimensión de los vectores:", info.config.params.vectors)
else:
    print("La colección configurada todavía no existe.")


Conexión con Qdrant establecida.

Colecciones existentes:
- data-academy
- documents
- tarea_final_ia

Colección configurada: tarea_final_ia
La colección ya existe.
Puntos almacenados: 0
Dimensión de los vectores: {'': VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=HnswConfigDiff(m=24, ef_construct=256, full_scan_threshold=None, max_indexing_threads=None, on_disk=None, memory=None, payload_m=24, inline_storage=None), quantization_config=None, on_disk=False, memory=None, datatype=<Datatype.FLOAT32: 'float32'>, multivector_config=None)}


Instalar y configurar el modelo de embeddings

In [18]:
%pip install -U openai

Note: you may need to restart the kernel to use updated packages.


Generar un embedding de prueba

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

texto_prueba = df_incidentes.iloc[0]["documento"]

respuesta = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=texto_prueba
)

vector_prueba = respuesta.data[0].embedding

print("Dimensión del embedding:", len(vector_prueba))
print("Primeros 5 valores:", vector_prueba[:5])


Dimensión del embedding: 1536
Primeros 5 valores: [-0.0272369384765625, 0.0634765625, 0.06146240234375, 0.020599365234375, -0.007671356201171875]


Cargar los incidentes en Qdrant

In [22]:

info = qdrant_client.get_collection(collection_name)

config_vectores = info.config.params.vectors

# La colección puede usar vectores sin nombre o con nombre.
if isinstance(config_vectores, dict):
    vector_config = config_vectores.get("")
    if vector_config is None:
        raise ValueError(
            "No se encontró el vector sin nombre en la colección."
        )
else:
    vector_config = config_vectores

if vector_config.size != 1536:
    raise ValueError(
        f"Dimensión incorrecta: {vector_config.size}. "
        "Se esperaba 1536."
    )

print("Configuración de vectores validada: 1536 dimensiones.")


Configuración de vectores validada: 1536 dimensiones.


Ejecuta nuevamente el código de carga

In [23]:

import os
from uuid import UUID

from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient, models

# 1. Cargar las variables de entorno
load_dotenv(override=True)

openai_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

qdrant_client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)

collection_name = os.environ["QDRANT_COLLECTION_NAME"]

# 2. Validar que existan datos para cargar
if df_incidentes.empty:
    raise ValueError("No hay incidentes para cargar.")

if "documento" not in df_incidentes.columns:
    raise ValueError(
        "Primero debes construir la columna 'documento'."
    )

if not qdrant_client.collection_exists(collection_name):
    raise ValueError(
        f"No existe la colección {collection_name}."
    )

# 3. Validar la dimensión de los vectores de Qdrant
info = qdrant_client.get_collection(collection_name)
config_vectores = info.config.params.vectors

if isinstance(config_vectores, dict):
    vector_config = config_vectores.get("")

    if vector_config is None:
        raise ValueError(
            "No se encontró el vector sin nombre en la colección."
        )
else:
    vector_config = config_vectores

if vector_config.size != 1536:
    raise ValueError(
        f"Dimensión incorrecta: {vector_config.size}. "
        "Se esperaba 1536."
    )

print("Configuración de vectores validada: 1536 dimensiones.")

# 4. Preparar los incidentes y sus documentos
registros = df_incidentes.to_dict("records")
documentos = [r["documento"] for r in registros]

puntos = []
tamano_lote = 50

# 5. Generar embeddings por lotes
for inicio in range(0, len(registros), tamano_lote):
    lote_registros = registros[inicio:inicio + tamano_lote]
    lote_documentos = documentos[inicio:inicio + tamano_lote]

    respuesta = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=lote_documentos
    )

    embeddings = sorted(
        respuesta.data,
        key=lambda x: x.index
    )

    # 6. Crear los puntos que se guardarán en Qdrant
    for registro, embedding in zip(lote_registros, embeddings):
        payload = {
            "incidente_id": str(registro["incidente_id"]),
            "codigo": registro["codigo"],
            "titulo": registro["titulo"],
            "componente": registro["componente"],
            "estado_incidente": registro["estado_incidente"],
            "documento": registro["documento"]
        }

        # Evitar guardar valores nulos
        payload = {
            k: v for k, v in payload.items()
            if v is not None
        }

        puntos.append(
            models.PointStruct(
                id=str(UUID(registro["incidente_id"])),
                vector=embedding.embedding,
                payload=payload
            )
        )

# 7. Insertar los puntos en Qdrant por lotes
for inicio in range(0, len(puntos), tamano_lote):
    qdrant_client.upsert(
        collection_name=collection_name,
        points=puntos[inicio:inicio + tamano_lote],
        wait=True
    )

# 8. Mostrar el resultado
print("Carga completada.")
print("Incidentes procesados:", len(puntos))
print(
    "Puntos en Qdrant:",
    qdrant_client.get_collection(collection_name).points_count
)


Configuración de vectores validada: 1536 dimensiones.
Carga completada.
Incidentes procesados: 15
Puntos en Qdrant: 15


Probar una búsqueda semántica

In [24]:

# 1. Definir la consulta
pregunta = (
    "La aplicación no puede encontrar el servicio por DNS "
    "y aparece el error UnknownHostException."
)

# 2. Generar el embedding de la consulta
respuesta = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=pregunta
)

vector_pregunta = respuesta.data[0].embedding

# 3. Buscar los incidentes más similares en Qdrant
resultado_busqueda = qdrant_client.query_points(
    collection_name=collection_name,
    query=vector_pregunta,
    limit=3,
    with_payload=True
)

# 4. Mostrar los resultados
for punto in resultado_busqueda.points:
    print("-" * 50)
    print("Código:", punto.payload.get("codigo"))
    print("Título:", punto.payload.get("titulo"))
    print("Componente:", punto.payload.get("componente"))
    print("Similitud:", round(punto.score, 4))
    print("Documento:", punto.payload.get("documento"))


--------------------------------------------------
Código: INC-0012
Título: Fallo de resolución DNS
Componente: Red
Similitud: 0.5686
Documento: Código: INC-0012

Título: Fallo de resolución DNS

Componente: Red

Descripción: Una aplicación no puede resolver el nombre de un servicio dependiente.

Mensaje de error: UnknownHostException

Estado del incidente: resuelto

Hipótesis:
Hipótesis: El registro DNS del servicio podría ser incorrecto o no estar disponible.
Estado: confirmada
Evidencia: La resolución del nombre falla desde el ambiente afectado.

Soluciones:
Solución: Verificar la resolución DNS desde el ambiente afectado y corregir el registro o la configuración pertinente.
Estado: aplicada
Resultado: efectiva
Hipótesis relacionada: El registro DNS del servicio podría ser incorrecto o no estar disponible.

Adjuntos:
Sin adjuntos registrados
--------------------------------------------------
Código: INC-0003
Título: Certificado TLS vencido
Componente: Conectividad segura
Similitud: 

Revisar los tres resultados

In [25]:

codigos = ["INC-0012", "INC-0003", "INC-0013"]

for codigo in codigos:
    resultados = qdrant_client.scroll(
        collection_name=collection_name,
        scroll_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="codigo",
                    match=models.MatchValue(value=codigo)
                )
            ]
        ),
        limit=1,
        with_payload=True,
        with_vectors=False
    )

    puntos, _ = resultados

    for punto in puntos:
        print("=" * 70)
        print("Código:", punto.payload.get("codigo"))
        print("Título:", punto.payload.get("titulo"))
        print("Componente:", punto.payload.get("componente"))
        print("Documento:", punto.payload.get("documento"))


Código: INC-0012
Título: Fallo de resolución DNS
Componente: Red
Documento: Código: INC-0012

Título: Fallo de resolución DNS

Componente: Red

Descripción: Una aplicación no puede resolver el nombre de un servicio dependiente.

Mensaje de error: UnknownHostException

Estado del incidente: resuelto

Hipótesis:
Hipótesis: El registro DNS del servicio podría ser incorrecto o no estar disponible.
Estado: confirmada
Evidencia: La resolución del nombre falla desde el ambiente afectado.

Soluciones:
Solución: Verificar la resolución DNS desde el ambiente afectado y corregir el registro o la configuración pertinente.
Estado: aplicada
Resultado: efectiva
Hipótesis relacionada: El registro DNS del servicio podría ser incorrecto o no estar disponible.

Adjuntos:
Sin adjuntos registrados
Código: INC-0003
Título: Certificado TLS vencido
Componente: Conectividad segura
Documento: Código: INC-0003

Título: Certificado TLS vencido

Componente: Conectividad segura

Descripción: Una comunicación entre 

Consultar las soluciones actualizadas en Neon

In [26]:

import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(override=True)

# 1. Obtener los códigos recuperados por Qdrant
codigos = [
    punto.payload["codigo"]
    for punto in resultado_busqueda.points
    if punto.payload and punto.payload.get("codigo")
]

if not codigos:
    raise ValueError("Qdrant no devolvió códigos de incidentes.")

print("Incidentes recuperados:", codigos)

# 2. Conectarse a Neon
engine_neon = create_engine(os.environ["NEON_DATABASE_URL"])

# 3. Consultar las soluciones registradas actualmente en Neon
parametros = {
    f"codigo_{i}": codigo
    for i, codigo in enumerate(codigos)
}

marcadores = ", ".join(
    f":codigo_{i}" for i in range(len(codigos))
)

consulta_sql = text(f"""
    SELECT
        i.codigo,
        i.titulo,
        i.componente,
        i.estado AS estado_incidente,
        s.descripcion AS solucion,
        s.estado AS estado_solucion,
        s.resultado AS resultado_solucion,
        s.fecha_creacion AS fecha_solucion
    FROM incidentes i
    LEFT JOIN soluciones s
        ON s.id_incidente = i.id
    WHERE i.codigo IN ({marcadores})
""")

df_soluciones = pd.read_sql_query(
    consulta_sql,
    engine_neon,
    params=parametros
)

# 4. Asociar cada incidente con su posición en la búsqueda
orden_relevancia = {
    codigo: posicion
    for posicion, codigo in enumerate(codigos, start=1)
}

df_soluciones["posicion_qdrant"] = (
    df_soluciones["codigo"].map(orden_relevancia)
)

# 5. Mostrar primero las soluciones efectivas,
# sin perder de vista la relevancia semántica
prioridad_resultado = {
    "efectiva": 1,
    "pendiente": 2,
    "no_efectiva": 3
}

df_soluciones["prioridad_solucion"] = (
    df_soluciones["resultado_solucion"]
    .map(prioridad_resultado)
    .fillna(4)
    .astype(int)
)

df_soluciones = df_soluciones.sort_values(
    ["prioridad_solucion", "posicion_qdrant"],
    ascending=[True, True]
)

display(df_soluciones)

# 6. Cerrar la conexión
engine_neon.dispose()


Incidentes recuperados: ['INC-0012', 'INC-0003', 'INC-0013']


,codigo,titulo,componente,estado_incidente,solucion,estado_solucion,resultado_solucion,fecha_solucion,posicion_qdrant,prioridad_solucion
1,INC-0012,Fallo de resolución DNS,Red,resuelto,Verificar la resolución DNS desde el ambiente ...,aplicada,efectiva,2026-10-12 17:15:40.584795+00:00,1,1
0,INC-0003,Certificado TLS vencido,Conectividad segura,resuelto,Renovar el certificado TLS mediante el procedi...,aplicada,efectiva,2026-08-20 13:43:05.946583+00:00,2,1
2,INC-0013,Error 403 al acceder a un recurso,Autorización,investigando,Revisar los permisos efectivos de la identidad...,propuesta,pendiente,2026-10-11 20:50:52.762489+00:00,3,2


In [27]:

def priorizar_soluciones(df_soluciones):
    """
    Ordena los incidentes recuperados para facilitar
    la selección de soluciones como antecedentes.
    """

    df = df_soluciones.copy()

    # Clasificar la evidencia disponible
    def clasificar_solucion(fila):
        resultado = fila["resultado_solucion"]
        estado = fila["estado_solucion"]

        if estado == "aplicada" and resultado == "efectiva":
            return "Solución aplicada y efectiva"

        if resultado == "no_efectiva":
            return "Solución no efectiva"

        if estado == "propuesta" or resultado == "pendiente":
            return "Solución pendiente de validar"

        return "Estado por revisar"

    df["clasificacion"] = df.apply(
        clasificar_solucion,
        axis=1
    )

    # Dar prioridad a la evidencia efectiva,
    # conservando la relevancia semántica como segundo criterio.
    prioridades = {
        "Solución aplicada y efectiva": 1,
        "Estado por revisar": 2,
        "Solución pendiente de validar": 3,
        "Solución no efectiva": 4
    }

    df["prioridad"] = df["clasificacion"].map(prioridades)

    return df.sort_values(
        ["prioridad", "posicion_qdrant"],
        ascending=[True, True]
    )

df_priorizadas = priorizar_soluciones(df_soluciones)

display(
    df_priorizadas[
        [
            "codigo",
            "titulo",
            "posicion_qdrant",
            "solucion",
            "estado_solucion",
            "resultado_solucion",
            "clasificacion"
        ]
    ]
)


,codigo,titulo,posicion_qdrant,solucion,estado_solucion,resultado_solucion,clasificacion
1,INC-0012,Fallo de resolución DNS,1,Verificar la resolución DNS desde el ambiente ...,aplicada,efectiva,Solución aplicada y efectiva
0,INC-0003,Certificado TLS vencido,2,Renovar el certificado TLS mediante el procedi...,aplicada,efectiva,Solución aplicada y efectiva
2,INC-0013,Error 403 al acceder a un recurso,3,Revisar los permisos efectivos de la identidad...,propuesta,pendiente,Solución pendiente de validar


Diagnosticar el resultado

In [28]:
print("Filas en df_soluciones:", len(df_soluciones))
print("Filas en df_priorizadas:", len(df_priorizadas))

print("\nColumnas disponibles:")
print(df_soluciones.columns.tolist())

print("\nContenido de df_priorizadas:")
display(df_priorizadas)

Filas en df_soluciones: 3
Filas en df_priorizadas: 3

Columnas disponibles:
['codigo', 'titulo', 'componente', 'estado_incidente', 'solucion', 'estado_solucion', 'resultado_solucion', 'fecha_solucion', 'posicion_qdrant', 'prioridad_solucion']

Contenido de df_priorizadas:


,codigo,titulo,componente,estado_incidente,solucion,estado_solucion,resultado_solucion,fecha_solucion,posicion_qdrant,prioridad_solucion,clasificacion,prioridad
1,INC-0012,Fallo de resolución DNS,Red,resuelto,Verificar la resolución DNS desde el ambiente ...,aplicada,efectiva,2026-10-12 17:15:40.584795+00:00,1,1,Solución aplicada y efectiva,1
0,INC-0003,Certificado TLS vencido,Conectividad segura,resuelto,Renovar el certificado TLS mediante el procedi...,aplicada,efectiva,2026-08-20 13:43:05.946583+00:00,2,1,Solución aplicada y efectiva,1
2,INC-0013,Error 403 al acceder a un recurso,Autorización,investigando,Revisar los permisos efectivos de la identidad...,propuesta,pendiente,2026-10-11 20:50:52.762489+00:00,3,2,Solución pendiente de validar,3


Generar una respuesta en lenguaje natural

In [29]:

def generar_respuesta(df_priorizadas):
    if df_priorizadas.empty:
        return "No se encontraron incidentes relacionados."

    efectivas = df_priorizadas[
        (df_priorizadas["estado_solucion"] == "aplicada")
        & (df_priorizadas["resultado_solucion"] == "efectiva")
    ]

    if efectivas.empty:
        return (
            "Se encontraron incidentes relacionados, "
            "pero no hay soluciones registradas como efectivas. "
            "Se recomienda revisar los casos antes de aplicar "
            "una solución."
        )

    lineas = [
        "Encontré antecedentes que podrían ayudar "
        "a investigar el incidente:",
        ""
    ]

    for _, fila in efectivas.iterrows():
        lineas.append(
            f"- {fila['codigo']}: {fila['titulo']}"
        )
        lineas.append(
            f"  Solución registrada: {fila['solucion']}"
        )
        lineas.append(
            f"  Resultado histórico: {fila['resultado_solucion']}"
        )
        lineas.append("")

    lineas.append(
        "Recomendación: verifica que las condiciones del incidente "
        "actual sean equivalentes antes de aplicar una solución. "
        "Los antecedentes no garantizan que funcione en este caso."
    )

    return "\n".join(lineas)


respuesta_agente = generar_respuesta(df_priorizadas)
print(respuesta_agente)


Encontré antecedentes que podrían ayudar a investigar el incidente:

- INC-0012: Fallo de resolución DNS
  Solución registrada: Verificar la resolución DNS desde el ambiente afectado y corregir el registro o la configuración pertinente.
  Resultado histórico: efectiva

- INC-0003: Certificado TLS vencido
  Solución registrada: Renovar el certificado TLS mediante el procedimiento aprobado y comprobar la cadena de confianza.
  Resultado histórico: efectiva

Recomendación: verifica que las condiciones del incidente actual sean equivalentes antes de aplicar una solución. Los antecedentes no garantizan que funcione en este caso.
